# Vector算子的设计、实现、编译与验证

## 概述

前面几节我们分别理解了Vector算子的计算模型（3.2）和数据切分策略（3.3）。本节把这些知识串成一条完整的**工程化开发流程**，遵循pyasc标准的四个阶段——**算子分析（设计）→ 核函数实现 → 编译运行 → 结果验证**。实现阶段采用`TPipe`/`TQue`框架：框架自动管理片上内存分配与双缓冲流水同步，开发者只需按CopyIn→Compute→CopyOut组织代码。

本节以Add算子为例讲解每个阶段的方法与**关键代码**，并在本节末尾给出Add算子的完整实现代码；下一节3.5章节实践将据此举一反三，独立开发ReLU算子。

### 学习目标

完成本节后，开发者应能够：

1. 掌握**算子分析（设计）**方法：明确数学表达式、输入输出、shape/format及所需接口，形成设计规格；
2. 掌握基于`TPipe`/`TQue`框架的**实现**方式与三段式编程范式；
3. 掌握Vector算子的**编译运行**参数；
4. 掌握基于torch的**结果验证**方法，并能将整套流程推广到其它Vector算子。

In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 算子分析（设计）

开发一个算子前，先做**算子分析**：明确数学表达式与计算逻辑、输入输出的数量与规格、核函数名称与参数，进而确定需要调用的pyasc接口。这一步的产出是一张**设计规格表**，是后续实现的依据。以Add算子为例：

1. **数学表达式与计算逻辑**：`z = x + y`。计算逻辑为——把数据从Global Memory搬入Local Memory，用矢量接口完成逐元素相加，再把结果搬回Global Memory（即CopyIn→Compute→CopyOut）。
2. **输入输出**：两个输入`x`、`y`，一个输出`z`；数据类型float，输出dtype与输入一致；shape为`(8, 2048)`，输出shape与输入相同；format为ND。
3. **核函数名称与参数**：核函数命名`vadd_kernel`，参数为`x`、`y`（输入）与`z`（输出）。
4. **所需接口**：数据搬运`asc.data_copy`、矢量加法`asc.add`、内存管理`asc.GlobalTensor`/`asc.LocalTensor`；采用框架实现时，片上内存分配与流水同步由`asc.TPipe`/`asc.TQue`统一管理（无需手写`set_flag`/`wait_flag`）。

由此得到Add算子的设计规格：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <tbody>
    <tr><td align="center">算子类型(OpType)</td><td colspan="4" align="center">Add</td></tr>
    <tr><td rowspan="3" align="center">算子输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
    <tr><td align="center">x</td><td align="center">(8, 2048)</td><td align="center">float</td><td align="center">ND</td></tr>
    <tr><td align="center">y</td><td align="center">(8, 2048)</td><td align="center">float</td><td align="center">ND</td></tr>
    <tr><td align="center">算子输出</td><td align="center">z</td><td align="center">(8, 2048)</td><td align="center">float</td><td align="center">ND</td></tr>
    <tr><td align="center">核函数名</td><td colspan="4" align="center">vadd_kernel</td></tr>
    <tr><td rowspan="3" align="center">使用的主要接口</td><td colspan="4" align="center">asc.data_copy：数据搬运接口</td></tr>
    <tr><td colspan="4" align="center">asc.add：矢量基础算术接口</td></tr>
    <tr><td colspan="4" align="center">asc.TPipe / asc.TQue：片上内存与流水同步管理</td></tr>
    <tr><td align="center">算子实现文件</td><td colspan="4" align="center">add_framework.py</td></tr>
  </tbody>
</table>

---
# 2. 核函数实现

### 2.1 TPipe/TQue框架

第2章2.6已通过手动版与框架版的对比说明了`TPipe`/`TQue`框架的定位：它把手动版中**片上内存申请、缓冲区编号管理、`set_flag`/`wait_flag`同步**等细节全部封装起来。本节展开框架的核心接口与用法：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">对象</th><th align="left">作用</th><th align="left">主要接口</th></tr>
  </thead>
  <tbody>
    <tr><td><code>asc.TPipe</code></td><td>片上内存管理器，为队列分配缓冲区</td><td><code>init_buffer</code></td></tr>
    <tr><td><code>asc.TQue</code></td><td>数据队列，自动管理双缓冲与流水同步</td><td><code>alloc_tensor</code> / <code>enque</code> / <code>deque</code> / <code>free_tensor</code></td></tr>
  </tbody>
</table>

队列按逻辑位置区分：输入队列位于`asc.TPosition.VECIN`，输出队列位于`asc.TPosition.VECOUT`。创建队列时指定缓冲区数量（`BUFFER_NUM=2`即双缓冲），框架据此自动完成缓冲区轮转与`set_flag`/`wait_flag`同步，开发者不再手写同步事件。

### 2.2 核函数骨架

核函数负责创建`TPipe`与输入/输出`TQue`、分配缓冲区，并在Tile循环中依次调用三段式函数。关键代码如下（省略GlobalTensor设置等重复片段）：

```python
@asc.jit(always_compile=True)
def vadd_kernel(x, y, z, block_length, tile_length: asc.ConstExpr[int]):
    offset = asc.get_block_idx() * block_length
    x_gm = asc.GlobalTensor(); y_gm = asc.GlobalTensor(); z_gm = asc.GlobalTensor()
    x_gm.set_global_buffer(x + offset)  # y_gm / z_gm 同理

    pipe = asc.TPipe()                                       # 片上内存管理器
    in_queue_x = asc.TQue(asc.TPosition.VECIN, BUFFER_NUM)   # 输入队列
    in_queue_y = asc.TQue(asc.TPosition.VECIN, BUFFER_NUM)
    out_queue_z = asc.TQue(asc.TPosition.VECOUT, BUFFER_NUM) # 输出队列
    pipe.init_buffer(in_queue_x, BUFFER_NUM, tile_length * x.dtype.sizeof())  # 分配缓冲区
    # in_queue_y / out_queue_z 同理

    for i in range(TILE_NUM * BUFFER_NUM):                   # 逐Tile循环
        copy_in(i, x_gm, y_gm, in_queue_x, in_queue_y, tile_length)
        compute(z_gm, in_queue_x, in_queue_y, out_queue_z, tile_length)
        copy_out(i, z_gm, out_queue_z, tile_length)
```

可以看到，核函数不再出现任何`set_flag`/`wait_flag`——同步交给了框架。

### 2.3 三段式Device函数

CopyIn、Compute、CopyOut拆成三个用`@asc.jit`修饰的Device函数，通过队列的`alloc_tensor`/`enque`/`deque`/`free_tensor`衔接。关键代码如下：

```python
@asc.jit
def copy_in(i, x_gm, y_gm, in_queue_x, in_queue_y, tile_length):
    x_local = in_queue_x.alloc_tensor(x_gm.dtype)           # 申请Tensor
    asc.data_copy(x_local, x_gm[i * tile_length:], tile_length)  # 搬入
    in_queue_x.enque(x_local)                               # 入队（y 同理）

@asc.jit
def compute(z_gm, in_queue_x, in_queue_y, out_queue_z, tile_length):
    x_local = in_queue_x.deque(z_gm.dtype)                  # 出队取输入
    y_local = in_queue_y.deque(z_gm.dtype)
    z_local = out_queue_z.alloc_tensor(z_gm.dtype)
    asc.add(z_local, x_local, y_local, tile_length)         # ★ 计算：换算子只改这一行
    out_queue_z.enque(z_local)
    in_queue_x.free_tensor(x_local); in_queue_y.free_tensor(y_local)  # 释放输入

@asc.jit
def copy_out(i, z_gm, out_queue_z, tile_length):
    z_local = out_queue_z.deque(z_gm.dtype)                 # 出队取结果
    asc.data_copy(z_gm[i * tile_length:], z_local, tile_length)  # 搬出
    out_queue_z.free_tensor(z_local)                        # 释放
```

三段式的价值在于：开发不同Vector算子时，**只需修改Compute中的计算接口**（如把`asc.add`换成单目的`asc.relu`），CopyIn/CopyOut完全复用。

---
# 3. 编译与运行

pyasc核函数通过`@asc.jit`装饰器触发JIT编译，在NPU上板模式下运行即可完成编译与执行：

In [ ]:
!python3 ./src/add_framework.py -r NPU

---
# 4. 结果验证

算子运行成功只说明程序跑通，还需验证**计算结果是否正确**。结果验证的方法——用torch原生运算生成参考结果、再用`torch.allclose`在容差范围内比对算子输出——已在第2章2.6详细介绍，这里不再重复。其核心代码为：

```python
z = vadd_launch(x, y)                 # 调用算子
assert torch.allclose(z, x + y)       # 与torch参考结果比对
```

需要强调的是：这套「torch准备输入 → 调用算子 → torch.allclose比对」的验证模式适用于所有Vector算子。开发新算子（如ReLU）时，只需把参考结果`x + y`换成对应的torch表达式（如`torch.relu(x)`），验证流程完全不变。

---
# 5. 小结

本节梳理了Vector算子工程化开发的完整四阶段：

- **算子分析（设计）**：明确数学表达式、输入输出、shape/format及所需接口，形成设计规格；
- **核函数实现**：`TPipe`管理片上内存、`TQue`自动完成双缓冲与流水同步；CopyIn/Compute/CopyOut拆成独立Device函数，开发新算子只改Compute；
- **编译运行**：`-r NPU`在NPU上板运行；
- **结果验证**：torch准备输入，`torch.allclose`比对算子输出与参考结果。

下一节3.5章节实践将举一反三，参照Add算子的开发流程独立开发ReLU算子。

---
# 6. Add算子完整实现代码

前面几节展示的都是关键代码片段，这里给出Add算子（`vadd_kernel`）的完整实现，方便对照阅读、也可作为3.5章节实践中ReLU算子开发的参考模板：

In [ ]:
!cat ./src/add_framework.py

---
## 课后练习

**选择题：**

1. 算子分析（设计）阶段主要明确以下哪些内容？
   - A. 只需确定启用的核数
   - B. 数学表达式、输入输出、shape/format及所需接口
   - C. 只需选定编译器优化级别
   - D. 只需确定算子实现文件名

2. 在`TPipe`/`TQue`框架中，负责为队列分配片上缓冲区的接口是？
   - A. `TQue.enque`
   - B. `TPipe.init_buffer`
   - C. `TQue.deque`
   - D. `asc.data_copy`

3. 使用框架方式实现时，与手动同步方式相比最主要的简化是？
   - A. 不需要切分数据
   - B. 不需要手写`set_flag`/`wait_flag`同步事件
   - C. 不需要CopyIn
   - D. 不需要torch验证

4. 输出队列应创建在哪个逻辑位置？
   - A. `asc.TPosition.VECIN`
   - B. `asc.TPosition.VECOUT`
   - C. `asc.TPosition.GM`
   - D. `asc.TPosition.A1`

5. 基于框架版Add开发框架版ReLU时，主要需要修改的是？
   - A. CopyIn函数
   - B. CopyOut函数
   - C. Compute函数中的计算接口
   - D. TPipe初始化

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.04_answer.txt